# No-pruning optimization pipeline

这个 Colab 只负责调用 VSCode/GitHub 里的 scripts。Pruning 已删除；先跑 `OriginalTopology_Adapt` 和 `RegionWiseOptimizedTopology_Adapt`。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!nvidia-smi

## 1. Clone 你的 GitHub repo

把下面的 GitHub URL 换成你自己的仓库。

In [ ]:
%cd /content
!rm -rf PhysTwin
!git clone -b zexu-no-pruning-opt https://github.com/zexuzang1/phystwin-neuspring-exp.git PhysTwin
%cd /content/PhysTwin
!ls

## 2. 安装依赖

如果某个包安装失败，先把报错发给我，不要继续跑训练。

In [ ]:
%cd /content/PhysTwin
!python -m pip install -r requirements_colab.txt

## 3. 解压 Drive 数据

In [ ]:
%cd /content/PhysTwin
!python scripts/unpack_drive_data.py \
  --drive-dir "/content/drive/MyDrive/PhysTwin_Data" \
  --project-dir "/content/PhysTwin" 

## 4. 打 PhysTwin patch

这一步允许 `train_warp.py` 使用外部 topology `.npz`，并关闭 headless Colab 里容易出错的可视化 / wandb。

In [ ]:
%cd /content/PhysTwin
!python scripts/patch_phystwin.py --project-dir "/content/PhysTwin" 

## 5. 运行无 pruning pipeline

会尽量复用你已有的 artifacts / checkpoints。

In [ ]:
%cd /content/PhysTwin
!python scripts/run_pipeline.py \
  --project-dir "/content/PhysTwin" \
  --case-name double_stretch_sloth \
  --base-path "/content/PhysTwin/data/different_types" \
  --original-topology "/content/PhysTwin/results/double_stretch_sloth_phystwin_topology_open3d.npz" \
  --reuse-artifact-zip "/content/drive/MyDrive/PhysTwin_Data/double_stretch_sloth_regionwise_optimization_artifacts.zip" \
  --reuse-checkpoint-zip "/content/drive/MyDrive/PhysTwin_Data/double_stretch_sloth_checkpoints.zip" \
  --reuse-checkpoints \
  --skip-existing-inference

## 6. 查看结果

In [ ]:
import pandas as pd
from pathlib import Path

adapt_root = Path("/content/PhysTwin/results/regionwise_optimization_final/double_stretch_sloth")
for name in ["base_adaptation_summary.csv", "topology_compare.csv", "cd_track.csv", "final_summary.csv"]:
    p = adapt_root / name
    print("\n===", p, "===")
    if p.exists():
        display(pd.read_csv(p))
    else:
        print("missing")

## 7. 打包保存回 Drive

In [ ]:
%cd /content/PhysTwin
!python scripts/pack_artifacts.py \
  --adapt-root "/content/PhysTwin/results/regionwise_optimization_final/double_stretch_sloth" \
  --out-zip "/content/drive/MyDrive/PhysTwin_Data/double_stretch_sloth_no_pruning_optimization_results.zip" 